# Práctica 5: Diccionarios, Tuplas y Conjuntos (Sets)

**Informática · Grado en Ingeniería Aeroespacial · UCLM**  
Curso 2026/27

> **Instrucciones:** Guarda una copia de este notebook en tu Google Drive (**Archivo > Guardar una copia en Drive**) y resuelve los ejercicios en las celdas indicadas. Cada ejercicio cuenta con celdas para tu solución y celdas de comprobación para verificar su correcto funcionamiento.

## Estructuras de Datos Complejas en Ingeniería Aeroespacial

Las listas de la práctica anterior resuelven series homogéneas de datos, pero la información de ingeniería suele venir estructurada: propiedades indexadas por el nombre del material, lecturas agrupadas por subsistema, inventarios de sistemas sin duplicados. En esta práctica trabajamos el Tema 4.3: tuplas como registros inmutables y su desempaquetado, la función `zip` para emparejar series, diccionarios como índice de claves y conjuntos para el análisis de redundancia.

## Parte 1: Tuplas y la función `zip`

Una tupla agrupa valores que forman parte de un mismo registro, como una posición de navegación `(nombre, latitud, longitud)`, y no puede modificarse tras crearse: es la estructura natural para datos fijos. El **desempaquetado** da nombre a cada campo directamente en la asignación o en el `for`, y `zip` empareja dos series sincronizadas elemento a elemento produciendo tuplas.

### Ejercicio 1: La ruta de navegación (tuplas, inmutabilidad y desempaquetado)

El plan de vuelo de un avión se define como una lista de *waypoints*. Cada *waypoint* es una **tupla** `(nombre, latitud_deg, longitud_deg)`: el nombre es un `str` y las coordenadas, dos `float`. Las coordenadas son datos de navegación fijos: por eso una tupla y no una lista.

```python
ruta = [
    ("SOTW", 41.75, -1.90),
    ("LOGR", 42.46, -2.44),
    ("VITOR", 42.88, -2.72),
    ("BASO", 43.29, -2.10),
    ("BIO", 43.30, -2.91),
]
```

**Tareas:**
1. Define la lista `ruta` con las cinco tuplas anteriores.
2. Recórrela con desempaquetado, `for nombre, lat, lon in ruta:`, e imprime cada *waypoint* en una línea.
3. Comprueba la inmutabilidad: intenta corregir la latitud del primer punto con `ruta[0][1] = 41.76` y observa el `TypeError` que lanza Python. Comenta esa línea para continuar: un elemento de una tupla no puede cambiarse; si necesitaras corregir un *waypoint*, tendrías que sustituir la tupla completa (`ruta[0] = (...)`), sin tocar las demás.
4. Define la función `rango_latitudes(ruta)` que devuelva la **tupla** `(latitud_minima, latitud_maxima)` de la ruta. *Pista:* construye con una comprehension la lista de latitudes y aplícale `min()` y `max()`.

In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: 5 waypoints · ruta[0] = ('SOTW', 41.75, -1.9) · rango de latitudes = (41.75, 43.3)
print("Waypoints de la ruta:", len(ruta))
print("Primero de la ruta:  ", ruta[0])
print("Rango de latitudes:  ", rango_latitudes(ruta))

---
### Ejercicio 2: Ensayo de flexión del ala (`zip`)

En un ensayo estático en banco de pruebas se aplica al ala una secuencia de cargas crecientes y se mide la deflexión de la punta en cada escalón. Las dos series están sincronizadas por posición: `cargas_kN[i]` y `deflexiones_mm[i]` pertenecen al mismo escalón del ensayo.

```python
cargas_kN = [0.0, 10.0, 20.0, 30.0, 40.0, 50.0]
deflexiones_mm = [0.0, 1.8, 3.7, 5.9, 10.2, 14.5]
```

Ambas son listas de `float`.

**Tareas:**
1. Define la función `emparejar_ensayo(cargas, deflexiones)` que devuelva la **lista de tuplas** `(carga, deflexion)` construida con `zip`. *Pista:* el objeto que devuelve `zip` es un iterador; envuélvelo en `list()` para poder imprimirlo.
2. Define la función `primera_deflexion_excesiva(cargas, deflexiones, limite_mm=8.0)` que devuelva la **tupla completa** `(carga, deflexion)` del primer escalón cuya deflexión supera estrictamente el límite, y `None` si ninguno lo supera. Recorre los pares con desempaquetado: `for carga, deflexion in zip(cargas, deflexiones):`.

*Nota:* la deflexión crece más que proporcionalmente con la carga: el ala se acerca a su límite elástico. Y si las listas tuvieran longitudes distintas, `zip` se detendría en la más corta (Tema 4.3).

In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: 6 pares desde (0.0, 0.0) hasta (50.0, 14.5) · primer escalón por encima de 8 mm → (40.0, 10.2) ·
# con límite de 20 mm → None
cargas_kN = [0.0, 10.0, 20.0, 30.0, 40.0, 50.0]
deflexiones_mm = [0.0, 1.8, 3.7, 5.9, 10.2, 14.5]

pares = emparejar_ensayo(cargas_kN, deflexiones_mm)
print("Pares carga-deflexión:", pares)
print("Primer escalón por encima de 8 mm:", primera_deflexion_excesiva(cargas_kN, deflexiones_mm, limite_mm=8.0))
print("Primer escalón por encima de 20 mm:", primera_deflexion_excesiva(cargas_kN, deflexiones_mm, limite_mm=20.0))

## Parte 2: Diccionarios

Un diccionario indexa por **clave** en lugar de por posición: `materiales["Inconel 718"]` dice qué es cada valor y el acceso es inmediato. Recorremos diccionarios con `.items()`, consultamos claves que pueden no existir con `.get()`, combinamos estructuras (diccionarios anidados, listas de lecturas, tuplas como valores) y terminamos con la **lista de diccionarios**, el formato habitual de una tabla de registros.

### Ejercicio 3: Base de datos de materiales aeroespaciales (diccionarios anidados)

El departamento de materiales mantiene un catálogo de aleaciones y compuestos. Cada material se indexa por su nombre y sus propiedades forman a su vez un diccionario:

```python
materiales = {
    "Aluminio 7075-T6": {"densidad_g_cm3": 2.81, "limite_elastico_mpa": 503, "resistencia_traccion_mpa": 572},
    "Titanio Ti-6Al-4V": {"densidad_g_cm3": 4.43, "limite_elastico_mpa": 880, "resistencia_traccion_mpa": 950},
    "Inconel 718": {"densidad_g_cm3": 8.19, "limite_elastico_mpa": 1100, "resistencia_traccion_mpa": 1375},
    "CFRP (Fibra Carbono)": {"densidad_g_cm3": 1.55, "limite_elastico_mpa": 780, "resistencia_traccion_mpa": 1500}
}
```

`materiales` es un diccionario de `str` → `dict`: a cada nombre de material le corresponde un diccionario con las claves `"densidad_g_cm3"`, `"limite_elastico_mpa"` y `"resistencia_traccion_mpa"` (las tres, números en las unidades que indica cada clave).

**Tareas:**
1. Define la función `calcular_resistencia_especifica(catalogo, nombre_material)` que devuelva un `float` con la relación entre el límite elástico y la densidad:
   $$R_e = \frac{\sigma_y}{\rho} \quad \left(\frac{\text{MPa}}{\text{g/cm}^3}\right)$$
   Si `nombre_material` no está en el catálogo, debe devolver `None`. *Pista:* `catalogo.get(nombre_material)` devuelve `None` si la clave no existe.
2. Define la función `material_mas_ligero(catalogo)` que devuelva el nombre (`str`) del material de menor densidad.
3. Define la función `material_optimo_resistencia_peso(catalogo)` que devuelva el nombre (`str`) del material con mayor resistencia específica. *Pista:* recorre `catalogo.items()` y reutiliza la función de la tarea 1.

In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: resistencia específica del Ti-6Al-4V ≈ 198.65 MPa/(g/cm³) · material inventado → None ·
# más ligero = CFRP (Fibra Carbono) · óptimo resistencia/peso = CFRP (Fibra Carbono)
materiales = {
    "Aluminio 7075-T6": {"densidad_g_cm3": 2.81, "limite_elastico_mpa": 503, "resistencia_traccion_mpa": 572},
    "Titanio Ti-6Al-4V": {"densidad_g_cm3": 4.43, "limite_elastico_mpa": 880, "resistencia_traccion_mpa": 950},
    "Inconel 718": {"densidad_g_cm3": 8.19, "limite_elastico_mpa": 1100, "resistencia_traccion_mpa": 1375},
    "CFRP (Fibra Carbono)": {"densidad_g_cm3": 1.55, "limite_elastico_mpa": 780, "resistencia_traccion_mpa": 1500}
}

print("Resistencia específica del Ti-6Al-4V:", calcular_resistencia_especifica(materiales, "Titanio Ti-6Al-4V"))
print("Resistencia específica de un material inventado:", calcular_resistencia_especifica(materiales, "Aleación irreal"))
print("Material más ligero:", material_mas_ligero(materiales))
print("Material óptimo resistencia/peso:", material_optimo_resistencia_peso(materiales))

---
### Ejercicio 4: Resumen de telemetría de subsistemas (diccionarios con valores compuestos)

Un satélite baja un paquete de telemetría con varias lecturas por subsistema: temperaturas en °C, presión del tanque de helio en bar y voltaje de los paneles en voltios.

```python
telemetria = {
    "termico_bateria": [21.5, 22.0, 22.8, 24.1, 23.5],
    "presion_tanque_helio": [280, 278, 275, 274, 272],
    "voltaje_paneles": [28.2, 28.1, 28.3, 27.9, 28.0],
    "temperatura_tobera": [620, 645, 680, 710, 695]
}
```

`telemetria` es un diccionario de `str` → lista de números: a cada subsistema le corresponde la lista ordenada de sus lecturas.

**Tareas:**
1. Define la función `resumen_telemetria(telemetria)` que devuelva un **nuevo diccionario** con las mismas claves y, como valor de cada una, la tupla `(media, maximo, minimo)` de sus lecturas, con los tres elementos redondeados a 2 decimales con `round()`.
2. Recorre el diccionario de entrada con `.items()` para disponer a la vez del subsistema y de sus lecturas.

*Nota:* el retorno vuelve a combinar estructuras: un diccionario cuyos valores son tuplas de números.

In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: termico_bateria (22.78, 24.1, 21.5) · presion_tanque_helio (275.8, 280, 272) ·
# voltaje_paneles (28.1, 28.3, 27.9) · temperatura_tobera (670.0, 710, 620)
telemetria = {
    "termico_bateria": [21.5, 22.0, 22.8, 24.1, 23.5],
    "presion_tanque_helio": [280, 278, 275, 274, 272],
    "voltaje_paneles": [28.2, 28.1, 28.3, 27.9, 28.0],
    "temperatura_tobera": [620, 645, 680, 710, 695]
}

resumen = resumen_telemetria(telemetria)
for subsistema, estadisticos in resumen.items():
    media, maximo, minimo = estadisticos
    print(f"{subsistema}: media = {media}, máximo = {maximo}, mínimo = {minimo}")

---
### Ejercicio 5: El catálogo orbital (lista de diccionarios)

El catálogo de objetos en órbita se guarda como una **lista de diccionarios**: cada elemento de la lista es un satélite y cada satélite es un diccionario con las claves `"nombre"` (`str`), `"pais"` (`str`), `"orbita"` (`str`), `"inclinacion_deg"` (`float`) y `"altitud_km"` (`int`):

```python
catalogo_satelites = [
    {"nombre": "Sentinel-2A", "pais": "ESA", "orbita": "SSO", "inclinacion_deg": 98.62, "altitud_km": 786},
    {"nombre": "Hubble", "pais": "NASA", "orbita": "LEO", "inclinacion_deg": 28.47, "altitud_km": 540},
    {"nombre": "Galileo-FOC FM24", "pais": "ESA", "orbita": "MEO", "inclinacion_deg": 56.00, "altitud_km": 23222},
    {"nombre": "Hispasat 30W-6", "pais": "España", "orbita": "GEO", "inclinacion_deg": 0.05, "altitud_km": 35786},
    {"nombre": "Paz", "pais": "España", "orbita": "SSO", "inclinacion_deg": 97.44, "altitud_km": 514},
    {"nombre": "ISS (Estación Espacial)", "pais": "Multinacional", "orbita": "LEO", "inclinacion_deg": 51.64, "altitud_km": 420}
]
```

**Tareas:**
1. Define la función `filtrar_satelites(catalogo, pais=None, orbita=None, min_inclinacion=None)` que devuelva la **lista de nombres** (`str`) de los satélites que cumplen **a la vez** todos los criterios indicados: ser del país `pais`, estar en la órbita `orbita` y tener una inclinación mayor o igual que `min_inclinacion`. Un criterio con valor `None` no se aplica.
2. Define la función `agrupar_por_orbita(catalogo)` que devuelva un diccionario donde las claves sean los tipos de órbita que aparecen en el catálogo (`"SSO"`, `"LEO"`, `"MEO"`, `"GEO"`) y los valores, listas con los nombres de los satélites de cada categoría.

*Pista:* para la tarea 2, parte de un diccionario vacío y, para cada satélite, añade su nombre a la lista de su órbita, creando la lista si es la primera vez que aparece esa clave.

In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: España → ['Hispasat 30W-6', 'Paz'] · SSO → ['Sentinel-2A', 'Paz'] ·
# inclinación ≥ 50° → ['Sentinel-2A', 'Galileo-FOC FM24', 'Paz', 'ISS (Estación Espacial)'] ·
# agrupación: LEO [Hubble, ISS] · MEO [Galileo-FOC FM24] · GEO [Hispasat 30W-6] · SSO [Sentinel-2A, Paz]
catalogo_satelites = [
    {"nombre": "Sentinel-2A", "pais": "ESA", "orbita": "SSO", "inclinacion_deg": 98.62, "altitud_km": 786},
    {"nombre": "Hubble", "pais": "NASA", "orbita": "LEO", "inclinacion_deg": 28.47, "altitud_km": 540},
    {"nombre": "Galileo-FOC FM24", "pais": "ESA", "orbita": "MEO", "inclinacion_deg": 56.00, "altitud_km": 23222},
    {"nombre": "Hispasat 30W-6", "pais": "España", "orbita": "GEO", "inclinacion_deg": 0.05, "altitud_km": 35786},
    {"nombre": "Paz", "pais": "España", "orbita": "SSO", "inclinacion_deg": 97.44, "altitud_km": 514},
    {"nombre": "ISS (Estación Espacial)", "pais": "Multinacional", "orbita": "LEO", "inclinacion_deg": 51.64, "altitud_km": 420}
]

print("Satélites de España:", filtrar_satelites(catalogo_satelites, pais="España"))
print("Satélites en órbita SSO:", filtrar_satelites(catalogo_satelites, orbita="SSO"))
print("Satélites con inclinación ≥ 50°:", filtrar_satelites(catalogo_satelites, min_inclinacion=50.0))
print("Distribución por tipo de órbita:", agrupar_por_orbita(catalogo_satelites))

## Parte 3: Conjuntos (sets)

Un conjunto guarda elementos **únicos y sin orden**: el mismo sistema no aparece dos veces y la pregunta `x in conjunto` se responde de forma inmediata. Las operaciones `union`, `intersection` y `difference` expresan en una sola línea preguntas de ingeniería como «¿qué sistemas están en los dos módulos?» o «¿qué sistemas críticos se han quedado sin respaldo?».

### Ejercicio 6: Redundancia de sistemas vitales (operaciones con sets)

En misiones tripuladas los sistemas vitales deben estar duplicados: presentes tanto en el Módulo de Mando $A$ como en el Módulo de Servicio $B$. Los tres datos del enunciado son conjuntos de cadenas (`set` de `str`):

```python
modulo_a = {"soporte_vital", "control_termico", "navegacion_inercial", "comunicaciones_uhf", "baterias_litio"}
modulo_b = {"soporte_vital", "control_termico", "propulsion_rcs", "comunicaciones_uhf", "paneles_solares"}
sistemas_criticos = {"soporte_vital", "control_termico", "navegacion_inercial", "comunicaciones_uhf"}
```

**Tareas:**
1. Define la función `analizar_redundancia(mod_a, mod_b, criticos)` que devuelva una **tupla de tres conjuntos**:
   * `catalogo_total`: los sistemas presentes en al menos un módulo (`union`).
   * `sistemas_redundantes`: los sistemas presentes en los dos módulos a la vez (`intersection`).
   * `criticos_sin_redundancia`: los sistemas de `criticos` que **no** están duplicados, es decir, `criticos.difference(sistemas_redundantes)`: carecen de respaldo.
2. Imprime cada resultado con `sorted()` para poder compararlos: un `set` no tiene un orden de impresión definido.

*Nota:* otra vez estructuras combinadas: la función devuelve una tupla cuyos elementos son conjuntos.

In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: inventario total → 7 sistemas · con respaldo dual → comunicaciones_uhf, control_termico y soporte_vital ·
# críticos sin respaldo → ['navegacion_inercial']
modulo_a = {"soporte_vital", "control_termico", "navegacion_inercial", "comunicaciones_uhf", "baterias_litio"}
modulo_b = {"soporte_vital", "control_termico", "propulsion_rcs", "comunicaciones_uhf", "paneles_solares"}
sistemas_criticos = {"soporte_vital", "control_termico", "navegacion_inercial", "comunicaciones_uhf"}

total, redundantes, sin_respaldo = analizar_redundancia(modulo_a, modulo_b, sistemas_criticos)
print("Inventario total:", sorted(total))
print("Con respaldo dual:", sorted(redundantes))
print("Críticos sin respaldo:", sorted(sin_respaldo))

---
### Ejercicio 7 (ampliación): Log de diagnóstico (dicts y sets juntos)

Ejercicio opcional, para casa o para quien termine antes. Al final de una misión, el ordenador de a bordo vuelca su log de diagnóstico: una **lista de mensajes** `"NIVEL subsistema"`, donde el nivel es `"OK"`, `"WARN"` o `"CRIT"`.

```python
log_diagnostico = [
    "OK soporte_vital",
    "WARN termico_bateria",
    "OK control_termico",
    "CRIT navegacion_inercial",
    "OK comunicaciones_uhf",
    "WARN navegacion_inercial",
    "OK propulsion_rcs",
    "WARN voltaje_paneles",
    "OK soporte_vital",
    "OK navegacion_inercial"
]
```

**Tareas:**
1. Define la función `contar_niveles(log)` que devuelva un diccionario nivel → número de mensajes de ese nivel. *Pista:* `mensaje.split()` separa el nivel del subsistema; para contar, usa el patrón contador con `.get()` del Tema 4.3: `conteo[nivel] = conteo.get(nivel, 0) + 1`.
2. Define la función `subsistemas_con_alertas(log)` que devuelva el conjunto de subsistemas con al menos un mensaje `"WARN"` o `"CRIT"`.
3. Define la función `subsistemas_sanos(log, criticos)` que, con `sistemas_criticos` del ejercicio anterior, devuelva el conjunto de sistemas críticos **sin** ninguna alerta (`difference`).

In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: {'OK': 6, 'WARN': 3, 'CRIT': 1} · alertas en navegacion_inercial, termico_bateria y voltaje_paneles ·
# críticos sanos → comunicaciones_uhf, control_termico y soporte_vital
log_diagnostico = [
    "OK soporte_vital",
    "WARN termico_bateria",
    "OK control_termico",
    "CRIT navegacion_inercial",
    "OK comunicaciones_uhf",
    "WARN navegacion_inercial",
    "OK propulsion_rcs",
    "WARN voltaje_paneles",
    "OK soporte_vital",
    "OK navegacion_inercial"
]
sistemas_criticos = {"soporte_vital", "control_termico", "navegacion_inercial", "comunicaciones_uhf"}

print("Mensajes por nivel:", contar_niveles(log_diagnostico))
print("Subsistemas con alertas:", sorted(subsistemas_con_alertas(log_diagnostico)))
print("Críticos sanos:", sorted(subsistemas_sanos(log_diagnostico, sistemas_criticos)))

---

**Departamento de Tecnologías y Sistemas de Información**  
Escuela de Ingeniería Industrial y Aeroespacial de Toledo — UCLM